In [14]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
import numpy as np
import pandas as pd
import h5py
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
from tqdm import tqdm 
from scipy import stats
from scipy.io import loadmat, savemat
from pathlib import Path
import glob
import cv2
import matplotlib.pyplot as plt


/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_26748/1031787245.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


In [3]:
#for loading the CaliAliAlignment file 
def _decode_matlab_string(ds):
    arr = ds[()]
    return "".join(chr(int(x)) for x in arr.ravel())

def _decode_matlab_scalar(ds):
    return ds[()].ravel()[0].item()

def load_caliai_input_files(mat_path):
    rows = []
    with h5py.File(mat_path, "r") as f:
        top_refs = f["CaliAli_options/inter_session_alignment/input_files"][()]
        
        for i in range(top_refs.shape[0]):
            row_ref = top_refs[i, 0]
            row_ds = f[row_ref]          # 5x1 cell row
            cell_refs = row_ds[()]
            
            input_file = _decode_matlab_string(f[cell_refs[0, 0]])
            col2 = _decode_matlab_scalar(f[cell_refs[1, 0]])
            col3 = _decode_matlab_scalar(f[cell_refs[2, 0]])
            n_frames = int(_decode_matlab_scalar(f[cell_refs[3, 0]]))
            det_file = _decode_matlab_string(f[cell_refs[4, 0]])
            
            rows.append({
                "input_file": input_file,
                "col2": col2,
                "col3": col3,
                "n_frames": n_frames,
                "det_file": det_file,
            })
    
    return pd.DataFrame(rows)

def trim_behav_multidf_to_caliai_lengths(
    behav_multidf_subset,
    check_df,
    aligned_file_col="aligned_file",
    target_len_col="ms_n_frames",
):
    if target_len_col not in check_df.columns:
        raise KeyError(
            f"{target_len_col!r} not in check_df columns. "
            f"Available columns: {list(check_df.columns)}"
        )

    target_lengths = (
        check_df[[aligned_file_col, target_len_col]]
        .dropna(subset=[target_len_col])
        .assign(**{target_len_col: lambda df: df[target_len_col].astype(int)})
        .drop_duplicates(subset=[aligned_file_col])
        .set_index(aligned_file_col)[target_len_col]
    )

    trimmed_parts = []
    summary_rows = []

    outer_files = behav_multidf_subset.index.get_level_values(0).unique()

    for aligned_file in outer_files:
        session_df = behav_multidf_subset.loc[aligned_file]
        current_n = len(session_df)

        if aligned_file not in target_lengths.index:
            summary_rows.append({
                "aligned_file": aligned_file,
                "original_rows": current_n,
                "target_rows": None,
                "rows_removed": None,
                "status": "missing_target_length",
            })
            continue

        target_n = int(target_lengths.loc[aligned_file])

        if current_n < target_n:
            summary_rows.append({
                "aligned_file": aligned_file,
                "original_rows": current_n,
                "target_rows": target_n,
                "rows_removed": None,
                "status": "current_shorter_than_target",
            })
            continue

        trimmed_df = session_df.iloc[:target_n].copy()
        trimmed_parts.append((aligned_file, trimmed_df))

        summary_rows.append({
            "aligned_file": aligned_file,
            "original_rows": current_n,
            "target_rows": target_n,
            "rows_removed": current_n - target_n,
            "status": "trimmed" if current_n > target_n else "unchanged",
        })

    if trimmed_parts:
        trimmed_multidf = pd.concat(
            [df for _, df in trimmed_parts],
            keys=[name for name, _ in trimmed_parts],
            names=behav_multidf_subset.index.names,
        )
    else:

        trimmed_multidf = behav_multidf_subset.copy()

    trim_summary_df = pd.DataFrame(summary_rows)
    return trimmed_multidf, trim_summary_df


In [4]:
## load and do some preprocessing on the CNMFE traces 
def normalize(trace, percentile=True):
    """ Normalize a fluorescence trace by its max or its 99th percentile. """
    trace = trace - np.min(trace)
    if np.percentile(trace, 99) > 0:
        if percentile:
            trace = trace / np.percentile(trace, 99)
        else:
            trace = trace / np.max(trace)
    return trace

def load_and_filter_traces(extract_mat_path: str,
                           labels_mat_path: str,
                           label_key: str = 'labels_ex'
                          ) -> pd.DataFrame:
    """
    Load temporal_weights from extract_mat_path (v7.3) and one of the
    labels_* arrays from labels_mat_path, keep only columns where
    labels == 1, and return as a pandas DataFrame.

    Parameters
    ----------
    extract_mat_path : str
        Path to the v7.3 .mat file containing an 'output/temporal_weights' dataset.
    labels_mat_path : str
        Path to the .mat file (v7.3 or earlier) containing a 'labels' group/struct.
    label_key : str, optional
        Which labels field to use: one of 'labels_ex', 'labels_ml', or
        'labels_overall'. Default is 'labels_ex'.

    Returns
    -------
    pd.DataFrame
        Rows = frames, columns = kept cells (named 'cell_<original_index>').
    """
    # validate choice
    allowed = ('labels_ex','labels_ml','labels_overall')
    if label_key not in allowed:
        raise ValueError(f"label_key must be one of {allowed}, got '{label_key}'")

    # 1) load & transpose temporal_weights → shape (nFrames, nCells)
    with h5py.File(extract_mat_path, 'r') as f:
        tw = f['output']['temporal_weights'][()]  # often (nCells, nFrames)
    tw = np.asarray(tw).T

    # 2) try to load chosen labels via HDF5; if that fails, fall back to loadmat
    try:
        with h5py.File(labels_mat_path, 'r') as f:
            hl = f['labels'][label_key][()]
    except OSError:
        mat = loadmat(labels_mat_path,
                      struct_as_record=False,
                      squeeze_me=True)
        lbl = mat['labels']  # either a dict or a mat_struct
        # pull out the right attribute/key
        if isinstance(lbl, dict):
            hl = lbl[label_key]
        else:
            hl = getattr(lbl, label_key)

    # 3) shape‐check & squeeze
    hl = np.asarray(hl).squeeze()
    if tw.shape[1] != hl.size:
        raise ValueError(
            f"dimension mismatch: temporal_weights is {tw.shape}, "
            f"labels array '{label_key}' has length {hl.size}"
        )

    # 4) filter & build DataFrame
    mask    = (hl == 1)
    kept_i  = np.nonzero(mask)[0]
    filtered = tw[:, mask]
    cols    = [f'cell_{i}' for i in kept_i]

    return (pd.DataFrame(filtered, columns=cols), hl)

def print_h5_tree(name, obj):
    """
    Callback for h5py.File.visititems.
    Prints group/dataset name and, for datasets, its shape and dtype.
    """
    if isinstance(obj, h5py.Group):
        print(f"Group:   {name}/")
    elif isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name}  — shape={obj.shape}, dtype={obj.dtype}")

    
def zScoreTraces(dirPath, CNMFE_real_cells, miniscopeFramesPerSecond):
    
    CNMFE_real_cells = CNMFE_real_cells.apply(pd.to_numeric, errors='coerce')
    C_normalized = CNMFE_real_cells.apply(lambda col: normalize(col), axis=0)
    C_z_scored = CNMFE_real_cells.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(CNMFE_real_cells)-1)*(1/miniscopeFramesPerSecond), len(CNMFE_real_cells)), unit='s'), drop=True)
    C_normalized_z_scored = C_normalized.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(C_normalized)-1)*(1/miniscopeFramesPerSecond), len(C_normalized)), unit='s'), drop=True)

    ##load spatial components by session
    # for v4 dimensions are 600x600 pixels
 
    C_normalized_z_scored.to_csv(dirPath+'_C_traces_filtered_origHz.csv')
    
    print('finished, saved:')
    print(dirPath+'_C_traces_filtered_origHz.csv')
    
    return(C_normalized_z_scored)

def zScoreTraces_withGappedTime(dirPath, CNMFE_real_cells, miniscopeFramesPerSecond, idxsList):
    # Convert all columns to numeric (as before)
    CNMFE_real_cells = CNMFE_real_cells.apply(pd.to_numeric, errors='coerce')
    # 1) Build a list of *all* original frame‐numbers (0…N−1),
    #    then remove the “excised” ranges to get `kept_frames`.
    #    N = M + total_removed
    removed_indices = []
    for start, end in idxsList:
        # remove [start, end) from original
        removed_indices.extend(range(start, end))
    removed_set = set(removed_indices)
    # total frames originally = len(kept) + len(removed)
    total_original_frames = len(CNMFE_real_cells) + len(removed_set)
    # generate all original indices 0…N−1
    all_indices = np.arange(total_original_frames)
    # keep only those not in removed_set, in sorted order
    kept_frames = np.array([i for i in all_indices if i not in removed_set])
    # sanity‐check: kept_frames.size == len(CNMFE_real_cells)
    if kept_frames.shape[0] != len(CNMFE_real_cells):
        raise ValueError(
            "Number of kept frames (%d) != number of rows in CNMFE_real_cells (%d)"
            % (kept_frames.shape[0], len(CNMFE_real_cells))
        )
    # 2) Create a "gapped" timedelta index from those kept_frames:
    #    time (in seconds) = frame_number * (1 / samplingRate)
    times_sec = kept_frames * (1.0 / miniscopeFramesPerSecond)
    time_index = pd.to_timedelta(times_sec, unit='s')
    # 3) Now normalize & z‐score (unchanged from before), but *do not* set the index yet:
    C_normalized = CNMFE_real_cells.apply(lambda col: (col - col.min()) / (col.max() - col.min()), axis=0)
    C_z_scored   = CNMFE_real_cells.apply(stats.zscore, axis=0)
    C_norm_z     = C_normalized.apply(stats.zscore, axis=0)
    # 4) Finally, assign our custom `time_index` to both:
    C_z_scored.index   = time_index
    C_norm_z.index     = time_index
    # 5) (Optional) save to CSV
    C_norm_z.to_csv(dirPath + '_C_traces_filtered_origHz.csv')
    print("Finished, saved to:", dirPath + '_C_traces_filtered_origHz.csv')
    return C_norm_z, C_z_scored

In [10]:

def load_labels_struct(labels_mat_path):
    try:
        with h5py.File(labels_mat_path, 'r') as f:
            g = f['labels']
            out = {}
            for k in ['labels_ex', 'labels_ml', 'labels_overall']:
                out[k] = np.asarray(g[k][()]).squeeze().astype(np.int16)
            return out
    except OSError:
        mat = loadmat(labels_mat_path, struct_as_record=False, squeeze_me=True)
        lbl = mat['labels']
        out = {}
        for k in ['labels_ex', 'labels_ml', 'labels_overall']:
            out[k] = np.asarray(getattr(lbl, k)).squeeze().astype(np.int16)
        return out


def merge_human_labels(labels_ex_a, labels_ex_b):
    if labels_ex_a.shape != labels_ex_b.shape:
        raise ValueError(f"Shape mismatch: {labels_ex_a.shape} vs {labels_ex_b.shape}")

    merged = np.zeros_like(labels_ex_a, dtype=np.int16)
    merged[(labels_ex_a == -1) | (labels_ex_b == -1)] = -1
    merged[(merged == 0) & ((labels_ex_a == 1) | (labels_ex_b == 1))] = 1
    return merged


def recompute_overall(merged_labels_ex, labels_ml):
    if merged_labels_ex.shape != labels_ml.shape:
        raise ValueError(f"Shape mismatch: {merged_labels_ex.shape} vs {labels_ml.shape}")

    labels_overall = labels_ml.astype(np.int16).copy()
    human_labeled = merged_labels_ex != 0
    labels_overall[human_labeled] = merged_labels_ex[human_labeled]
    return labels_overall


In [5]:
# for calculating event rate 

def binarize_traces(df, threshold):
    """
    Return a copy of df where every value < threshold becomes 0,
    and every value >= threshold becomes 1.
    """
    return (df >= threshold).astype(int)

def suprathreshold_to_events(binary_series):
    b = np.asarray(binary_series).astype(int)
    # rising edges: 0->1 transitions
    events = (b == 1) & (np.r_[0, b[:-1]] == 0)
    return events.astype(int)

def get_event_rate(df, samplingRate):
    """
    Given a binarized DataFrame `df` of shape (n_frames, n_cells),
    compute, for each cell (column), the number of “events” in each
    forward-looking 1-second window. Returns a DataFrame of shape
    (n_frames - samplingRate + 1, n_cells).
    """
    # 1) grab the raw values (frames × cells)
    arr = df.values.astype(int)  
    # 2) build a “boxcar” kernel of length = samples per second
    kernel = np.ones(samplingRate, dtype=int)
    # 3) convolve along the time-axis for each cell (axis=0)
    #    mode='valid' gives you only positions where the full window fits
    rates = np.apply_along_axis(
        lambda col: np.convolve(col, kernel, mode='valid'),
        axis=0,
        arr=arr
    )
    # 4) build a new index so that row i corresponds to window df.index[i : i+samplingRate]
    #    if your original df.index is RangeIndex starting at 0, this is simply 0..n-sR
    new_index = df.index[: rates.shape[0]]
    return pd.DataFrame(rates, index=new_index, columns=df.columns)

def get_event_rate_same_length(df, samplingRate, pad_value=0):
    """
    Forward-looking 1-second (samplingRate-frame) window sums,
    same behavior as your mode='valid' version, but returns length == len(df)
    by padding the END with pad_value before convolution.
    """
    arr = df.to_numpy(dtype=int)
    kernel = np.ones(samplingRate, dtype=int)

    # pad the end so that 'valid' produces N outputs
    pad = samplingRate - 1
    arr_pad = np.pad(arr, ((0, pad), (0, 0)), mode='constant', constant_values=pad_value)

    rates = np.apply_along_axis(lambda col: np.convolve(col, kernel, mode='valid'), axis=0, arr=arr_pad)
    return pd.DataFrame(rates, index=df.index, columns=df.columns)

In [6]:
# velocity analysis on the behavior data 

def add_velocity_from_xy(
    df,
    cmPerPixel,
    x_col="X_coor",
    y_col="Y_coor",
    timestamp_col=None,
    sampling_period_s=None,
    downsample_rules=("100ms", "200ms"),
    median_filter_velocity=False,
    median_filter_window_samples=None,
    spatial_filter_velocity=False,
    spatial_filter_distance_cm=1.0,
    spatial_filter_future_window_s=1.0,
):
    """
    Add Distance (cm) and Velocity (cm/s) from X/Y coordinates.

    Optional post-processing:
    - median filter on velocity
    - spatial filter: if centroid does not move more than
      spatial_filter_distance_cm at any point over the next
      spatial_filter_future_window_s seconds, set velocity to 0
      for that frame.
    """
    def _future_displacement_keep_mask(
        x_vals,
        y_vals,
        cm_per_pixel,
        timestamp_ms=None,
        sampling_period=None,
        future_window_s=1.0,
        threshold_cm=1.0,
    ):
        x_vals = np.asarray(x_vals, dtype=float)
        y_vals = np.asarray(y_vals, dtype=float)
        n = len(x_vals)
        keep = np.zeros(n, dtype=bool)

        if timestamp_ms is not None:
            t_vals = pd.to_numeric(pd.Series(timestamp_ms), errors="coerce").to_numpy(dtype=float)
            window_ms = future_window_s * 1000.0

            for i in range(n):
                if np.isnan(x_vals[i]) or np.isnan(y_vals[i]) or np.isnan(t_vals[i]):
                    continue

                j_end = np.searchsorted(t_vals, t_vals[i] + window_ms, side="right")
                if j_end <= i + 1:
                    continue

                dx = x_vals[i + 1:j_end] - x_vals[i]
                dy = y_vals[i + 1:j_end] - y_vals[i]
                valid = ~(np.isnan(dx) | np.isnan(dy))
                if not valid.any():
                    continue

                max_disp_cm = np.sqrt(dx[valid] ** 2 + dy[valid] ** 2).max() * cm_per_pixel
                keep[i] = max_disp_cm > threshold_cm

        else:
            if sampling_period is None:
                raise ValueError("Provide timestamp_ms or sampling_period for spatial filter")

            future_window_samples = max(1, int(np.ceil(future_window_s / sampling_period)))

            for i in range(n):
                if np.isnan(x_vals[i]) or np.isnan(y_vals[i]):
                    continue

                j_end = min(n, i + 1 + future_window_samples)
                if j_end <= i + 1:
                    continue

                dx = x_vals[i + 1:j_end] - x_vals[i]
                dy = y_vals[i + 1:j_end] - y_vals[i]
                valid = ~(np.isnan(dx) | np.isnan(dy))
                if not valid.any():
                    continue

                max_disp_cm = np.sqrt(dx[valid] ** 2 + dy[valid] ** 2).max() * cm_per_pixel
                keep[i] = max_disp_cm > threshold_cm

        return keep

    def _apply_spatial_filter(
        velocity_series,
        x_vals,
        y_vals,
        cm_per_pixel,
        timestamp_ms=None,
        sampling_period=None,
        future_window_s=1.0,
        threshold_cm=1.0,
    ):
        keep = _future_displacement_keep_mask(
            x_vals=x_vals,
            y_vals=y_vals,
            cm_per_pixel=cm_per_pixel,
            timestamp_ms=timestamp_ms,
            sampling_period=sampling_period,
            future_window_s=future_window_s,
            threshold_cm=threshold_cm,
        )

        out_vel = velocity_series.astype(float).copy()
        out_vel[(~keep) & out_vel.notna()] = 0.0
        return out_vel

    out = df.copy()

    dx = out[x_col].diff()
    dy = out[y_col].diff()

    step_distance_px = np.sqrt(dx**2 + dy**2)
    out["Distance"] = step_distance_px * cmPerPixel

    if timestamp_col is not None:
        dt_s = pd.to_numeric(out[timestamp_col], errors="coerce").diff() / 1000.0
    elif sampling_period_s is not None:
        dt_s = pd.Series(sampling_period_s, index=out.index, dtype=float)
        dt_s.iloc[0] = np.nan
    else:
        raise ValueError("Provide either timestamp_col or sampling_period_s")

    out["Velocity"] = out["Distance"] / dt_s

    velocity_source_col = "Velocity"

    if median_filter_velocity:
        if median_filter_window_samples is None:
            raise ValueError("Provide median_filter_window_samples when median_filter_velocity=True")
        out["Velocity_filtered"] = (
            out["Velocity"]
            .rolling(window=median_filter_window_samples, center=True, min_periods=1)
            .median()
        )
        velocity_source_col = "Velocity_filtered"

    if spatial_filter_velocity:
        out["Velocity_spatial_filtered"] = _apply_spatial_filter(
            velocity_series=out[velocity_source_col],
            x_vals=out[x_col],
            y_vals=out[y_col],
            cm_per_pixel=cmPerPixel,
            timestamp_ms=out[timestamp_col] if timestamp_col is not None else None,
            sampling_period=sampling_period_s if timestamp_col is None else None,
            future_window_s=spatial_filter_future_window_s,
            threshold_cm=spatial_filter_distance_cm,
        )

    if timestamp_col is not None:
        td_index = pd.to_timedelta(pd.to_numeric(out[timestamp_col], errors="coerce"), unit="ms")
    else:
        td_index = pd.to_timedelta(np.arange(len(out)) * sampling_period_s, unit="s")

    out_td = out.set_index(td_index, drop=False)

    downsampled = {}
    for rule in downsample_rules:
        ds = out_td.resample(rule).bfill()
        ds[[x_col, y_col]] = out_td[[x_col, y_col]].resample(rule).mean()

        dx_ds = ds[x_col].diff()
        dy_ds = ds[y_col].diff()
        ds["Distance"] = np.sqrt(dx_ds**2 + dy_ds**2) * cmPerPixel

        sampling_s = pd.to_timedelta(rule).total_seconds()
        ds["Velocity"] = ds["Distance"] / sampling_s

        ds_velocity_source_col = "Velocity"

        if median_filter_velocity:
            ds["Velocity_filtered"] = (
                ds["Velocity"]
                .rolling(window=median_filter_window_samples, center=True, min_periods=1)
                .median()
            )
            ds_velocity_source_col = "Velocity_filtered"

        if spatial_filter_velocity:
            ds["Velocity_spatial_filtered"] = _apply_spatial_filter(
                velocity_series=ds[ds_velocity_source_col],
                x_vals=ds[x_col],
                y_vals=ds[y_col],
                cm_per_pixel=cmPerPixel,
                sampling_period=sampling_s,
                future_window_s=spatial_filter_future_window_s,
                threshold_cm=spatial_filter_distance_cm,
            )

        downsampled[rule] = ds

    return out, downsampled

## event rate by speed bins 

def avg_event_rate_by_speed_bins(df, idx_by_bin, speed_col='linearSpeedcmPerSecond'):
    """
    Given:
      • df            : DataFrame with one column per cell (instantaneous rates)
                        plus a `speed_col`.
      • idx_by_bin    : dict mapping (low, high) tuples → list of row-indices
                        where speed ∈ [low, high) (or >= low if high is inf).
      • speed_col     : name of the speed column in df (will be dropped).
    Returns:
      • avg_df        : DataFrame indexed by your (low, high) bins,
                        columns are the cell-names, entries are the mean
                        event-rate of that cell over all frames in that bin.
    """
    # 1) Identify all “cell” columns (everything except the speed column)
    event_cols = [c for c in df.columns if c != speed_col]

    # 2) Make a DataFrame to hold means; use a MultiIndex of your bin tuples
    bin_index = pd.MultiIndex.from_tuples(idx_by_bin.keys(), names=['low','high'])
    avg_df = pd.DataFrame(index=bin_index, columns=event_cols, dtype=float)

    # 3) For each bin, pull those rows and take the column‐wise mean
    for bin_range, idxs in idx_by_bin.items():
        if len(idxs)>0:
            avg_df.loc[bin_range] = df.loc[idxs, event_cols].mean()
        else:
            avg_df.loc[bin_range] = np.nan

    return avg_df


In [7]:
# velocity validation videos 

def create_velocity_validation_videos(
    tracking_dir,
    gcamp_df,
    source_col="ezTrackOutput",
    frame_col="closestBehavCamFrameIdx",
    x_col="X_coor",
    y_col="Y_coor",
    velocity_col="Velocity",
    output_dir=None,
    text_mode="pad_right",   # "pad_right" or "overlay"
    pad_width=240,
    codec="MJPG",
    marker_size=18,
    marker_thickness=2,
    font_scale=0.65,
    text_thickness=2,
):
    tracking_dir = Path(tracking_dir)
    output_dir = Path(output_dir) if output_dir is not None else tracking_dir / "velocity_validation_videos"
    output_dir.mkdir(parents=True, exist_ok=True)

    if text_mode not in {"pad_right", "overlay"}:
        raise ValueError("text_mode must be 'pad_right' or 'overlay'")

    def unique_path(path):
        path = Path(path)
        if not path.exists():
            return path
        k = 2
        while True:
            candidate = path.with_name(f"{path.stem}_v{k}{path.suffix}")
            if not candidate.exists():
                return candidate
            k += 1

    results = []

    grouped = gcamp_df.groupby(source_col, dropna=False)

    for eztrack_src, grp in tqdm(grouped, total=gcamp_df[source_col].nunique(), desc="Velocity validation videos"):
        if pd.isna(eztrack_src):
            continue

        csv_name = Path(str(eztrack_src)).name
        video_name = csv_name.replace("_LocationOutput.csv", ".avi")
        input_vid = tracking_dir / video_name

        if not input_vid.exists():
            results.append({
                "ezTrackOutput": eztrack_src,
                "video_path": str(input_vid),
                "output_path": None,
                "status": "missing_video",
            })
            continue

        overlay = (
            grp[[frame_col, x_col, y_col, velocity_col]]
            .dropna(subset=[frame_col])
            .assign(**{frame_col: lambda d: d[frame_col].astype(int)})
            .groupby(frame_col, as_index=True)
            .agg({
                x_col: "mean",
                y_col: "mean",
                velocity_col: "mean",
            })
            .sort_index()
        )

        cap = cv2.VideoCapture(str(input_vid))
        fps = cap.get(cv2.CAP_PROP_FPS)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        overlay = overlay.reindex(range(n_frames))

        suffix = "_velocity_validation_padded.avi" if text_mode == "pad_right" else "_velocity_validation_overlay.avi"
        output_vid = unique_path(output_dir / video_name.replace(".avi", suffix))

        out_w = w + pad_width if text_mode == "pad_right" else w
        out_h = h

        fourcc = cv2.VideoWriter_fourcc(*codec)
        writer = cv2.VideoWriter(str(output_vid), fourcc, fps, (out_w, out_h), True)

        font = cv2.FONT_HERSHEY_SIMPLEX
        margin = 12
        line_gap = 28

        for frame_idx in range(n_frames):
            ret, frame = cap.read()
            if not ret:
                break

            if frame.ndim == 2:
                frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)

            x_val = overlay.iloc[frame_idx][x_col]
            y_val = overlay.iloc[frame_idx][y_col]
            vel_val = overlay.iloc[frame_idx][velocity_col]

            if text_mode == "pad_right":
                canvas = np.zeros((h, out_w, 3), dtype=np.uint8)
                canvas[:, :w] = frame
            else:
                canvas = frame.copy()

            if pd.notna(x_val) and pd.notna(y_val):
                xm = int(np.clip(round(x_val), 0, w - 1))
                ym = int(np.clip(round(y_val), 0, h - 1))
                cv2.drawMarker(
                    canvas,
                    (xm, ym),
                    (255, 255, 255),
                    markerType=cv2.MARKER_TILTED_CROSS,
                    markerSize=marker_size,
                    thickness=marker_thickness,
                    line_type=cv2.LINE_AA,
                )

            line1 = f"Vel: {vel_val:.2f} cm/s" if pd.notna(vel_val) else "Vel: nan"
            line2 = f"X: {x_val:.1f} px" if pd.notna(x_val) else "X: nan"
            line3 = f"Y: {y_val:.1f} px" if pd.notna(y_val) else "Y: nan"

            if text_mode == "pad_right":
                text_x = w + margin
                text_y = margin + 20
            else:
                widths = [cv2.getTextSize(t, font, font_scale, text_thickness)[0][0] for t in (line1, line2, line3)]
                text_x = max(margin, w - max(widths) - margin)
                text_y = margin + 20

            for i, text in enumerate((line1, line2, line3)):
                y_text = text_y + i * line_gap
                if text_mode == "overlay":
                    cv2.putText(
                        canvas, text, (text_x, y_text), font, font_scale,
                        (0, 0, 0), text_thickness + 2, cv2.LINE_AA
                    )
                cv2.putText(
                    canvas, text, (text_x, y_text), font, font_scale,
                    (255, 255, 255), text_thickness, cv2.LINE_AA
                )

            writer.write(canvas)

        cap.release()
        writer.release()

        results.append({
            "ezTrackOutput": eztrack_src,
            "video_path": str(input_vid),
            "output_path": str(output_vid),
            "text_mode": text_mode,
            "status": "ok",
        })

    return pd.DataFrame(results)



In [12]:
#calcium analysis data - extract and ActSort labels 
dirPath = '/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays/m311/20260420_001027_953/'
GCaMPData_EXTRACT_fName = r'day010_combined_ds_mc_Aligned_20260420_001027_953.mat'
#have two label files here, need to merge 
ActSortLabels_fName_0 = r'precomputed_output_LABELS.mat'
ActSortLabels_fName_1 = r'precomputed_output_LABELS_0.mat'

In [16]:
# if needed to merge labels 
merged_out_name = 'precomputed_output_LABELS_merged.mat'

labels_path_0 = str(Path(dirPath) / ActSortLabels_fName_0)
labels_path_1 = str(Path(dirPath) / ActSortLabels_fName_1)
merged_out_path = str(Path(dirPath) / merged_out_name)
L0 = load_labels_struct(labels_path_0)
L1 = load_labels_struct(labels_path_1)
merged_labels_ex = merge_human_labels(L0['labels_ex'], L1['labels_ex'])
# use model labels from the first file by default
# if you want, you can check they are identical before trusting this
if not np.array_equal(L0['labels_ml'], L1['labels_ml']):
    print("Warning: labels_ml differs between files; using labels_ml from first file.")
merged_labels_ml = L0['labels_ml'].copy()
merged_labels_overall = recompute_overall(merged_labels_ex, merged_labels_ml)
merged_labels = {
    'labels_ex': merged_labels_ex,
    'labels_ml': merged_labels_ml,
    'labels_overall': merged_labels_overall,
}
savemat(merged_out_path, {'labels': merged_labels}, do_compression=True)
print("Saved merged labels to:", merged_out_path)
print("Merged counts:")
print("labels_ex  good:", np.sum(merged_labels_ex == 1),
      "bad:", np.sum(merged_labels_ex == -1),
      "unlabeled:", np.sum(merged_labels_ex == 0))
print("labels_overall good:", np.sum(merged_labels_overall == 1),
      "bad:", np.sum(merged_labels_overall == -1))
ActSortLabels_fName = merged_out_name

Saved merged labels to: /Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays/m311/20260420_001027_953/precomputed_output_LABELS_merged.mat
Merged counts:
labels_ex  good: 656 bad: 62 unlabeled: 0
labels_overall good: 656 bad: 62


In [17]:
#'labels_ex', 'labels_ml', or 'labels_overall'. Default is 'labels_ex'
GCAMP_traces_humanSorted, labels_human = load_and_filter_traces(dirPath+GCaMPData_EXTRACT_fName, dirPath+ActSortLabels_fName, 'labels_ex')
print('traces loaded: human labels')
GCAMP_traces_modelSorted, labels_model = load_and_filter_traces(dirPath+GCaMPData_EXTRACT_fName, dirPath+ActSortLabels_fName, 'labels_ml')
print('traces loaded: ML labels')
GCAMP_traces_OvrallSorted, labels_overall = load_and_filter_traces(dirPath+GCaMPData_EXTRACT_fName, dirPath+ActSortLabels_fName, 'labels_overall')
print('traces loaded')

traces loaded: human labels
traces loaded: ML labels
traces loaded


In [19]:
# if data was "removed" from the calcium signals for the EXTRACT video add that information here, if no data was removed leave as empty list
# videosRemoved = [] 
videosRemoved = []; 
videoLength = 1000; 
# video 1 would 0:1000, 
idxsList=[((vidStart*1000)-1000,(vidStart)*1000) for vidStart in videosRemoved]

miniscopeFramesPerSecond = 20

GCAMP_traces_Zscore = zScoreTraces(dirPath, GCAMP_traces_OvrallSorted, miniscopeFramesPerSecond)
print('loaded all traces')

if len(videosRemoved)>0:
    GCAMP_traces_ZscoreNormalized_Gapped, GCAMP_traces_Zscore_Gapped  = zScoreTraces_withGappedTime(dirPath, GCAMP_traces_OvrallSorted, miniscopeFramesPerSecond, idxsList)
else: 
    GCAMP_traces_Zscore_Gapped = GCAMP_traces_Zscore
    

finished, saved:
/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays/m311/20260420_001027_953/_C_traces_filtered_origHz.csv
loaded all traces


In [22]:
#CaliAli alignment 
mat_path = "/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays/m311/day010_combined_ds_mc_Aligned.mat"
input_files_df = load_caliai_input_files(mat_path)


In [23]:
len(GCAMP_traces_Zscore_Gapped)

271420

In [29]:
#behavior analysis info 
pathToAlignedBehavData = "/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackEzTrack/aligned_miniscope_to_eztrack_311/"
base = Path(pathToAlignedBehavData)

files = sorted(base.glob("*AlignedToEzTrack.csv"))

behav_multidf = pd.concat(
    [pd.read_csv(f) for f in files],
    keys=[f.name for f in files],
    names=["source_file", "row_in_file"]
)

behav_multidf.head()

Frame Number  \
source_file                                     row_in_file                 
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                       0   
                                                1                       1   
                                                2                       2   
                                                3                       3   
                                                4                       4   

                                                             Time Stamp (ms)  \
source_file                                     row_in_file                    
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                         -4   
                                                1                         52   
                                                2                         99   
                                                3                        152   
                                                4                        201   

                                                             Buffer Index  \
source_file                                     row_in_file                 
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                       0   
                                                1                       0   
                                                2                       0   
                                                3                       0   
                                                4                       0   

                                                             sys_clock_BehavCamFrame  \
source_file                                     row_in_file                            
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                                -17   
                                                1                                 47   
                                                2                                112   
                                                3                                159   
                                                4                                224   

                                                             closestBehavCamFrameIdx  \
source_file                                     row_in_file                            
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                                  0   
                                                1                                  1   
                                                2                                  2   
                                                3                                  3   
                                                4                                  4   

                                                             abs_timestamp_diff_ms  \
source_file                                     row_in_file                          
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                               13   
                                                1                                5   
                                                2                               13   
                                                3                                7   
                                                4                               23   

                                                                 X_coor  \
source_file                                     row_in_file               
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0            640.117215   
                                                1            637.458656   
                                                2            636.436593   
                                                3            631.556153   
                                                4            628.991345   

                                                         

In [30]:
len(behav_multidf)

779766

In [28]:
input_files_df['input_file'].values

array(['/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day001_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day002_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day003_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day004_18_35_11_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day004_19_12_36_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day005_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day006_19_43_52_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day006_20_10_39_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day007_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_com

In [35]:
behav_multidfMiniscopeSubset = behav_multidf.loc[['02042025_19_15_29_miniscopeAlignedToEzTrack.csv', 
                                                  '03042025_15_39_01_miniscopeAlignedToEzTrack.csv', 
                                                  '04042025_21_05_03_miniscopeAlignedToEzTrack.csv',
                                                  '05042025_18_35_11_miniscopeAlignedToEzTrack.csv',
                                                  '05042025_19_12_36_miniscopeAlignedToEzTrack.csv',
                                                  '06042025_18_46_48_miniscopeAlignedToEzTrack.csv',                                               
                                                  '07042025_19_43_52_miniscopeAlignedToEzTrack.csv', 
                                                  '07042025_20_10_39_miniscopeAlignedToEzTrack.csv',
                                                  '08042025_16_21_42_miniscopeAlignedToEzTrack.csv',
                                                  '09042025_15_28_20_miniscopeAlignedToEzTrack.csv',
                                                  '10042025_16_39_41_miniscopeAlignedToEzTrack.csv',
                                                  '11042025_20_13_24_miniscopeAlignedToEzTrack.csv']]

In [36]:
len(behav_multidfMiniscopeSubset)

271420

In [45]:
## check that rows of outer levels of behav_multidfMiniscopeSubset match miniscope time stamps file
base_dir = Path("/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackEzTrack")
miniscope_ts_dir = base_dir / "timeStampsMiniscope_copied_311"
results = []
outer_files = behav_multidfMiniscopeSubset.index.get_level_values(0).unique()
for aligned_file in outer_files:
    # example: 01052025_19_58_29_miniscopeAlignedToEzTrack.csv
    session = aligned_file.replace("_miniscopeAlignedToEzTrack.csv", "")

    dd = session[2:4]
    mm = session[:2]
    yyyy = session[4:8]
    time_part = session[9:]  # after MMDDYYYY_

    miniscope_name = f"{yyyy}_{dd}_{mm}_311_{time_part}_timeStampsMiniscope.csv"
    miniscope_path = miniscope_ts_dir / miniscope_name

    aligned_n = len(behav_multidfMiniscopeSubset.loc[aligned_file])

    if miniscope_path.exists():
        miniscope_n = len(pd.read_csv(miniscope_path))
    else:
        miniscope_n = None

    results.append({
        "aligned_file": aligned_file,
        "miniscope_timestamp_file": miniscope_name,
        "aligned_rows": aligned_n,
        "miniscope_rows": miniscope_n,
        "match": aligned_n == miniscope_n if miniscope_n is not None else False,
        "timestamp_file_exists": miniscope_path.exists(),
    })
check_df = pd.DataFrame(results)
check_df


,aligned_file,miniscope_timestamp_file,aligned_rows,miniscope_rows,match,timestamp_file_exists
0,02042025_19_15_29_miniscopeAlignedToEzTrack.csv,2025_04_02_311_19_15_29_timeStampsMiniscope.csv,23700,23700,True,True
1,03042025_15_39_01_miniscopeAlignedToEzTrack.csv,2025_04_03_311_15_39_01_timeStampsMiniscope.csv,23692,23692,True,True
2,04042025_21_05_03_miniscopeAlignedToEzTrack.csv,2025_04_04_311_21_05_03_timeStampsMiniscope.csv,23693,23693,True,True
3,05042025_18_35_11_miniscopeAlignedToEzTrack.csv,2025_04_05_311_18_35_11_timeStampsMiniscope.csv,15691,15691,True,True
4,05042025_19_12_36_miniscopeAlignedToEzTrack.csv,2025_04_05_311_19_12_36_timeStampsMiniscope.csv,23693,23693,True,True
5,06042025_18_46_48_miniscopeAlignedToEzTrack.csv,2025_04_06_311_18_46_48_timeStampsMiniscope.csv,23688,23688,True,True
6,07042025_19_43_52_miniscopeAlignedToEzTrack.csv,2025_04_07_311_19_43_52_timeStampsMiniscope.csv,23692,23692,True,True
7,07042025_20_10_39_miniscopeAlignedToEzTrack.csv,2025_04_07_311_20_10_39_timeStampsMiniscope.csv,18810,18810,True,True
8,08042025_16_21_42_miniscopeAlignedToEzTrack.csv,2025_04_08_311_16_21_42_timeStampsMiniscope.csv,23691,23691,True,True
9,09042025_15_28_20_miniscopeAlignedToEzTrack.csv,2025_04_09_311_15_28_20_timeStampsMiniscope.csv,23689,23689,True,True


In [46]:
input_files_df['input_file'].values

array(['/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day001_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day002_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day003_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day004_18_35_11_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day004_19_12_36_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day005_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day006_19_43_52_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day006_20_10_39_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_combined/combined_mc/day007_combined_ds_mc.mat',
       '/scratch/jma819/CaliAli_testData/m311_Yiwen_com

In [47]:
## correct for where fewer miniscope frames per day were aligned 

# make a working copy
input_files_df2 = input_files_df.copy()
check_df2 = check_df.copy()

input_files_df2["ms_session_key"] = (
    input_files_df2["input_file"]
    .astype(str)
    .str.extract(r"(?:^|/)(\d{4}_\d{2}_\d{2}_\d+_\d{2}_\d{2}_\d{2})(?:/|$)")[0]
)



check_df2["ms_session_key"] = (
    check_df2["miniscope_timestamp_file"]
    .str.extract(r"(\d{4}_\d{2}_\d{2}_\d+_\d{2}_\d{2}_\d{2})_timeStampsMiniscope\.csv")[0]
)

ms_lookup = input_files_df2[["ms_session_key", "input_file", "n_frames"]].rename(
    columns={
        "input_file": "ms_input_file",
        "n_frames": "ms_n_frames"
    }
)

check_df_aligned = check_df2.merge(ms_lookup, on="ms_session_key", how="left")


check_df_aligned


,aligned_file,miniscope_timestamp_file,aligned_rows,miniscope_rows,match,timestamp_file_exists,ms_session_key,ms_input_file,ms_n_frames
0,02042025_19_15_29_miniscopeAlignedToEzTrack.csv,2025_04_02_311_19_15_29_timeStampsMiniscope.csv,23700,23700,True,True,2025_04_02_311_19_15_29,NaN,NaN
1,03042025_15_39_01_miniscopeAlignedToEzTrack.csv,2025_04_03_311_15_39_01_timeStampsMiniscope.csv,23692,23692,True,True,2025_04_03_311_15_39_01,NaN,NaN
2,04042025_21_05_03_miniscopeAlignedToEzTrack.csv,2025_04_04_311_21_05_03_timeStampsMiniscope.csv,23693,23693,True,True,2025_04_04_311_21_05_03,NaN,NaN
3,05042025_18_35_11_miniscopeAlignedToEzTrack.csv,2025_04_05_311_18_35_11_timeStampsMiniscope.csv,15691,15691,True,True,2025_04_05_311_18_35_11,NaN,NaN
4,05042025_19_12_36_miniscopeAlignedToEzTrack.csv,2025_04_05_311_19_12_36_timeStampsMiniscope.csv,23693,23693,True,True,2025_04_05_311_19_12_36,NaN,NaN
5,06042025_18_46_48_miniscopeAlignedToEzTrack.csv,2025_04_06_311_18_46_48_timeStampsMiniscope.csv,23688,23688,True,True,2025_04_06_311_18_46_48,NaN,NaN
6,07042025_19_43_52_miniscopeAlignedToEzTrack.csv,2025_04_07_311_19_43_52_timeStampsMiniscope.csv,23692,23692,True,True,2025_04_07_311_19_43_52,NaN,NaN
7,07042025_20_10_39_miniscopeAlignedToEzTrack.csv,2025_04_07_311_20_10_39_timeStampsMiniscope.csv,18810,18810,True,True,2025_04_07_311_20_10_39,NaN,NaN
8,08042025_16_21_42_miniscopeAlignedToEzTrack.csv,2025_04_08_311_16_21_42_timeStampsMiniscope.csv,23691,23691,True,True,2025_04_08_311_16_21_42,NaN,NaN
9,09042025_15_28_20_miniscopeAlignedToEzTrack.csv,2025_04_09_311_15_28_20_timeStampsMiniscope.csv,23689,23689,True,True,2025_04_09_311_15_28_20,NaN,NaN


In [48]:
behav_multidfMiniscopeSubset.head()

Frame Number  \
source_file                                     row_in_file                 
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                       0   
                                                1                       1   
                                                2                       2   
                                                3                       3   
                                                4                       4   

                                                             Time Stamp (ms)  \
source_file                                     row_in_file                    
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                         -4   
                                                1                         52   
                                                2                         99   
                                                3                        152   
                                                4                        201   

                                                             Buffer Index  \
source_file                                     row_in_file                 
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                       0   
                                                1                       0   
                                                2                       0   
                                                3                       0   
                                                4                       0   

                                                             sys_clock_BehavCamFrame  \
source_file                                     row_in_file                            
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                                -17   
                                                1                                 47   
                                                2                                112   
                                                3                                159   
                                                4                                224   

                                                             closestBehavCamFrameIdx  \
source_file                                     row_in_file                            
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                                  0   
                                                1                                  1   
                                                2                                  2   
                                                3                                  3   
                                                4                                  4   

                                                             abs_timestamp_diff_ms  \
source_file                                     row_in_file                          
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0                               13   
                                                1                                5   
                                                2                               13   
                                                3                                7   
                                                4                               23   

                                                                 X_coor  \
source_file                                     row_in_file               
02042025_19_15_29_miniscopeAlignedToEzTrack.csv 0            640.117215   
                                                1            637.458656   
                                                2            636.436593   
                                                3            631.556153   
                                                4            628.991345   

                                                         

In [49]:
behav_multidfMiniscopeSubset_trimmed, trim_summary_df = trim_behav_multidf_to_caliai_lengths(
    behav_multidfMiniscopeSubset,
    check_df_aligned
)

trim_summary_df

,aligned_file,original_rows,target_rows,rows_removed,status
0,02042025_19_15_29_miniscopeAlignedToEzTrack.csv,23700,None,None,missing_target_length
1,03042025_15_39_01_miniscopeAlignedToEzTrack.csv,23692,None,None,missing_target_length
2,04042025_21_05_03_miniscopeAlignedToEzTrack.csv,23693,None,None,missing_target_length
3,05042025_18_35_11_miniscopeAlignedToEzTrack.csv,15691,None,None,missing_target_length
4,05042025_19_12_36_miniscopeAlignedToEzTrack.csv,23693,None,None,missing_target_length
5,06042025_18_46_48_miniscopeAlignedToEzTrack.csv,23688,None,None,missing_target_length
6,07042025_19_43_52_miniscopeAlignedToEzTrack.csv,23692,None,None,missing_target_length
7,07042025_20_10_39_miniscopeAlignedToEzTrack.csv,18810,None,None,missing_target_length
8,08042025_16_21_42_miniscopeAlignedToEzTrack.csv,23691,None,None,missing_target_length
9,09042025_15_28_20_miniscopeAlignedToEzTrack.csv,23689,None,None,missing_target_length


In [50]:
len(behav_multidfMiniscopeSubset_trimmed)

271420

In [51]:
len(GCAMP_traces_Zscore_Gapped)

271420

In [52]:
# concat behavior to GCamp data 
behav_cols = (
    behav_multidfMiniscopeSubset_trimmed[["X_coor", "Y_coor", "Distance_px", "ezTrackOutput", "closestBehavCamFrameIdx"]]
    .reset_index(drop=True)
)
gcamp_df = GCAMP_traces_Zscore_Gapped.reset_index(drop=True)
GCAMP_traces_Zscore_Gapped_with_behav = pd.concat(
    [gcamp_df, behav_cols],
    axis=1
)



In [53]:
GCAMP_traces_Zscore_Gapped_with_behav.head()

,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,...,cell_712,cell_713,cell_714,cell_716,cell_717,X_coor,Y_coor,Distance_px,ezTrackOutput,closestBehavCamFrameIdx
0,-0.613440,-0.078008,-0.695869,-0.626669,-0.713137,-0.092954,-0.629677,-0.676739,-0.664425,0.447824,...,-0.694291,0.890504,0.212217,-0.701491,-0.487638,640.117215,20.667540,0.000000,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,0
1,-0.365839,-0.491324,0.878930,-0.626669,-0.713137,-0.608581,3.165282,-0.676739,1.078922,-0.462309,...,0.310239,0.860607,0.134441,-0.701491,-0.149108,637.458656,21.062280,2.687705,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,1
2,-0.613440,-0.491324,0.018058,0.141768,-0.689096,2.160694,-0.629677,-0.501444,0.286478,0.068954,...,1.347648,-0.619424,2.099403,-0.701491,-0.487638,636.436593,22.075925,1.439475,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2
3,-0.613440,-0.491324,-0.695869,0.874475,-0.713137,-0.179605,-0.345774,-0.676739,-0.620516,-0.462309,...,0.082134,-0.619424,-0.663712,-0.701491,-0.487638,631.556153,22.297691,4.885476,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,3
4,0.107865,-0.491324,-0.695869,-0.626669,-0.713137,2.182898,0.175791,-0.676739,0.729127,-0.462309,...,0.263295,-0.619424,1.211605,0.485786,-0.487638,628.991345,22.307303,2.564826,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,4


In [54]:
cmPerPixel = 0.22

GCAMP_with_velocity, downsampled = add_velocity_from_xy(
    GCAMP_traces_Zscore_Gapped_with_behav,
    cmPerPixel=cmPerPixel,
    sampling_period_s=0.05,
    median_filter_velocity=True,
    median_filter_window_samples=30,
    spatial_filter_velocity=True,
    spatial_filter_distance_cm=1.0,
    spatial_filter_future_window_s=2.0,
)


In [55]:
# Velocity in Cm/Sec 
GCAMP_with_velocity.head()

,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,...,cell_717,X_coor,Y_coor,Distance_px,ezTrackOutput,closestBehavCamFrameIdx,Distance,Velocity,Velocity_filtered,Velocity_spatial_filtered
0,-0.613440,-0.078008,-0.695869,-0.626669,-0.713137,-0.092954,-0.629677,-0.676739,-0.664425,0.447824,...,-0.487638,640.117215,20.667540,0.000000,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,0,NaN,NaN,27.238494,27.238494
1,-0.365839,-0.491324,0.878930,-0.626669,-0.713137,-0.608581,3.165282,-0.676739,1.078922,-0.462309,...,-0.149108,637.458656,21.062280,2.687705,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,1,0.591295,11.825902,28.267333,28.267333
2,-0.613440,-0.491324,0.018058,0.141768,-0.689096,2.160694,-0.629677,-0.501444,0.286478,0.068954,...,-0.487638,636.436593,22.075925,1.439475,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,2,0.316685,6.333692,27.238494,27.238494
3,-0.613440,-0.491324,-0.695869,0.874475,-0.713137,-0.179605,-0.345774,-0.676739,-0.620516,-0.462309,...,-0.487638,631.556153,22.297691,4.885476,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,3,1.074805,21.496092,26.209654,26.209654
4,0.107865,-0.491324,-0.695869,-0.626669,-0.713137,2.182898,0.175791,-0.676739,0.729127,-0.462309,...,-0.487638,628.991345,22.307303,2.564826,/Volumes/fsmresfiles/Basic_Sciences/Phys/Contr...,4,0.564262,11.285236,24.272701,24.272701


In [56]:
## pull out velocity "bin" periods 
velocity_bins = [(0, 0.5), (0.5, 2), (2, 5), (5, 10), (10, np.inf)]
velocity_bin_ilocs = {(lo, hi): np.flatnonzero(((GCAMP_with_velocity["Velocity_spatial_filtered"] >= lo) & ((GCAMP_with_velocity["Velocity_spatial_filtered"] < hi) if np.isfinite(hi) else True)).to_numpy()).tolist() for lo, hi in velocity_bins}


In [57]:
## create velocity validation videos 

validation_summary = create_velocity_validation_videos(
    tracking_dir="/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackEzTrack/BehavCamConcactenated_326/rotated_and_cropped_avi",
    gcamp_df=GCAMP_with_velocity, velocity_col="Velocity_spatial_filtered"
)

Velocity validation videos: 100%|███████████████| 12/12 [00:00<00:00, 22.38it/s]


In [58]:
velocity_bin_ilocs

{(0, 0.5): [104,
  105,
  106,
  446,
  448,
  449,
  450,
  451,
  452,
  453,
  454,
  455,
  456,
  457,
  458,
  459,
  647,
  648,
  649,
  650,
  651,
  652,
  655,
  659,
  660,
  661,
  662,
  663,
  664,
  665,
  666,
  667,
  668,
  669,
  670,
  671,
  672,
  673,
  674,
  676,
  677,
  678,
  681,
  682,
  684,
  686,
  687,
  688,
  689,
  690,
  691,
  692,
  693,
  694,
  695,
  696,
  697,
  698,
  699,
  700,
  704,
  705,
  710,
  711,
  712,
  715,
  716,
  717,
  718,
  719,
  721,
  731,
  735,
  887,
  888,
  889,
  890,
  891,
  892,
  893,
  939,
  940,
  941,
  942,
  943,
  944,
  945,
  946,
  947,
  948,
  949,
  950,
  951,
  952,
  953,
  954,
  955,
  956,
  957,
  958,
  959,
  960,
  961,
  962,
  963,
  964,
  965,
  966,
  967,
  968,
  969,
  970,
  971,
  972,
  973,
  974,
  975,
  976,
  1056,
  1057,
  1058,
  1459,
  1460,
  1461,
  1462,
  1463,
  1464,
  1496,
  1497,
  1654,
  1655,
  1656,
  2054,
  2055,
  2056,
  2057,
  2058,
  2059,
  20

In [59]:
## calculate event rate
aligned_GCAMP = GCAMP_with_velocity.loc[:, GCAMP_with_velocity.columns.str.startswith('cell_')]

In [71]:
aligned_GCAMP

,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,...,cell_707,cell_708,cell_709,cell_710,cell_711,cell_712,cell_713,cell_714,cell_716,cell_717
0,-0.613440,-0.078008,-0.695869,-0.626669,-0.713137,-0.092954,-0.629677,-0.676739,-0.664425,0.447824,...,-0.308596,0.055504,-0.614436,-0.575761,0.640957,-0.694291,0.890504,0.212217,-0.701491,-0.487638
1,-0.365839,-0.491324,0.878930,-0.626669,-0.713137,-0.608581,3.165282,-0.676739,1.078922,-0.462309,...,0.145576,-0.652931,-0.614436,-0.197687,2.197652,0.310239,0.860607,0.134441,-0.701491,-0.149108
2,-0.613440,-0.491324,0.018058,0.141768,-0.689096,2.160694,-0.629677,-0.501444,0.286478,0.068954,...,-0.600332,-0.376901,-0.346739,1.496608,-0.632927,1.347648,-0.619424,2.099403,-0.701491,-0.487638
3,-0.613440,-0.491324,-0.695869,0.874475,-0.713137,-0.179605,-0.345774,-0.676739,-0.620516,-0.462309,...,-0.600332,-0.652931,-0.614436,-0.658161,-0.130726,0.082134,-0.619424,-0.663712,-0.701491,-0.487638
4,0.107865,-0.491324,-0.695869,-0.626669,-0.713137,2.182898,0.175791,-0.676739,0.729127,-0.462309,...,-0.600332,-0.323362,2.184204,2.342105,-0.632927,0.263295,-0.619424,1.211605,0.485786,-0.487638
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271415,-0.480875,-0.491324,-0.628242,-0.626669,-0.713137,-0.608581,-0.094443,0.124300,2.102102,-0.462309,...,0.520795,-0.652931,-0.608887,-0.658161,1.388318,1.573908,-0.619424,-0.516324,1.835712,-0.487638
271416,-0.613440,-0.373973,-0.094604,-0.626669,1.448705,-0.608581,-0.629677,-0.590303,1.137757,-0.462309,...,-0.509116,-0.652931,1.040085,-0.658161,-0.091657,1.501796,-0.555555,-0.426189,-0.701491,-0.487638
271417,-0.020350,0.670449,0.019937,0.918343,0.414260,0.178136,-0.120941,-0.676739,1.041156,-0.462309,...,-0.600332,1.606576,0.331115,-0.658161,-0.383991,-0.694291,-0.619424,1.336993,-0.641498,-0.487638
271418,-0.613440,-0.491324,-0.272511,0.648641,1.209023,-0.608581,-0.629677,-0.676739,3.369241,-0.462309,...,0.376648,0.331212,-0.574401,-0.658161,-0.632927,-0.694291,-0.619424,0.494158,0.450283,-0.229357


In [61]:
firingThresholdSD = 2.5
signalPeaks = binarize_traces(aligned_GCAMP, firingThresholdSD)
signalPeaksOnsets = signalPeaks.apply(suprathreshold_to_events, axis=0)

In [62]:
signalPeaksOnsets.head()

,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,...,cell_707,cell_708,cell_709,cell_710,cell_711,cell_712,cell_713,cell_714,cell_716,cell_717
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [65]:
# get the event rate per cell 
instantaneousEventRate = get_event_rate_same_length(signalPeaks, samplingRate=20).reset_index(drop=True)
instantaneousEventRateOnsets = get_event_rate_same_length(signalPeaksOnsets, samplingRate=20).reset_index(drop=True)

instantaneousEventRate["Velocity_spatial_filtered"] = GCAMP_with_velocity["Velocity_spatial_filtered"].reset_index(drop=True)
instantaneousEventRateOnsets["Velocity_spatial_filtered"] = GCAMP_with_velocity["Velocity_spatial_filtered"].reset_index(drop=True)

In [66]:
#avg rates (all above threshold) by speed 
avg_rates = avg_event_rate_by_speed_bins(
    instantaneousEventRate,
    velocity_bin_ilocs,
    speed_col="Velocity_spatial_filtered"
)
#avg rates (onsets - rising edge ) by speed 
avg_rates_onsets = avg_event_rate_by_speed_bins(
    instantaneousEventRateOnsets,
    velocity_bin_ilocs,
    speed_col="Velocity_spatial_filtered"
)

In [73]:
# save GCAMP_with_velocity, instantaneousEventRate, instantaneousEventRateOnsets
saveDir = Path('/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays/m311_analysis')
saveDir.mkdir(parents=True, exist_ok=True)

GCAMP_with_velocity.to_csv(saveDir / "GCAMP_with_velocity.csv", index=True)
instantaneousEventRate.to_csv(saveDir / "instantaneousEventRate.csv", index=True)
instantaneousEventRateOnsets.to_csv(saveDir / "instantaneousEventRateOnsets.csv", index=True)

In [67]:
instantaneousEventRate.head()

,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,...,cell_708,cell_709,cell_710,cell_711,cell_712,cell_713,cell_714,cell_716,cell_717,Velocity_spatial_filtered
0,0,0,1,0,0,0,1,0,0,0,...,0,1,1,0,0,0,0,1,0,27.238494
1,0,0,1,0,0,0,1,0,0,0,...,0,1,1,0,0,0,0,1,0,28.267333
2,0,0,1,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,1,0,27.238494
3,0,0,1,0,0,0,0,0,0,0,...,0,1,1,0,0,0,1,1,0,26.209654
4,0,0,1,0,0,0,0,0,0,0,...,0,1,1,0,0,0,1,1,0,24.272701


In [68]:
avg_rates.mean(axis=1)

low   high
0.0   0.5     0.371674
0.5   2.0     0.413413
2.0   5.0     0.504726
5.0   10.0    0.519655
10.0  inf     0.595953
dtype: float64

In [69]:
avg_rates_onsets.mean(axis=1)

low   high
0.0   0.5     0.254548
0.5   2.0     0.276039
2.0   5.0     0.322734
5.0   10.0    0.327601
10.0  inf     0.358298
dtype: float64